# Sparse autoencoders on known latent features

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/sparse_autoencoder.ipynb)

This notebook runs on CPU. It generates all data and performs the experiment; no model download or API credentials are needed. GPU timing is optional for attention.

Dependencies: Python 3.10+, torch, numpy, scipy, matplotlib. In Colab these are normally installed. If needed run `%pip install torch numpy scipy matplotlib` before the imports.

Tested on Python 3.12 with PyTorch 2.14.0+rocm10.1.0, NumPy 2.5.3, SciPy 1.18.1 and Matplotlib 3.11.2. The ROCm suffix identifies the development machine build; retain the PyTorch build supplied by Colab for its CPU or NVIDIA runtime. Different library versions can change timings and floating-point details.

In [ ]:
import math, time
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
torch.set_num_threads(4)

## Implementation

The functions below are the complete implementation. Fixed seeds make the inputs repeatable; timings depend on the machine. Read the accompanying tutorial for derivations and limits.

In [ ]:
class SparseAE(nn.Module):
    def __init__(self,d=8,width=24):
      super().__init__();self.encoder=nn.Linear(d,width);self.decoder=nn.Linear(width,d,bias=False)
      with torch.no_grad(): self.decoder.weight.copy_(F.normalize(self.decoder.weight,dim=0))
    def forward(self,x):
      z=F.relu(self.encoder(x));return self.decoder(z),z

def sparse_data(seed,n=4096):
    g=torch.Generator().manual_seed(seed)
    dictionary=F.normalize(torch.randn(12,8,generator=g),dim=1)
    features=(torch.rand(n,12,generator=g)<.08)*(.5+torch.rand(n,12,generator=g))
    return features@dictionary,features,dictionary

def sae_experiment():
    start=time.perf_counter();x,features,d=sparse_data(118);train=x[:3072];test=x[3072:]
    records=[];traces=[];best=None
    for seed in [0,1,2]:
      for penalty in [0.,.01,.05,.15]:
        torch.manual_seed(seed);model=SparseAE();opt=torch.optim.Adam(model.parameters(),lr=.003);history=[]
        for step in range(700):
          idx=torch.randint(len(train),(256,));recon,z=model(train[idx]);mse=(recon-train[idx]).square().mean();loss=mse+penalty*z.mean()
          opt.zero_grad();loss.backward();opt.step()
          with torch.no_grad():model.decoder.weight.copy_(F.normalize(model.decoder.weight,dim=0))
          if step%50==0:history.append([step,float(mse.detach()),float(z.detach().mean())])
        with torch.no_grad():
          recon,z=model(test);cos=d@F.normalize(model.decoder.weight,dim=0);rr,cc=linear_sum_assignment(-cos.abs().numpy());matched=cos.abs()[rr,cc]
          intervention=model.decoder.weight[:,int(cc[0])];alignment=float(F.cosine_similarity(intervention,d[0],dim=0).abs())
          rec=dict(seed=seed,penalty=penalty,mse=float((recon-test).square().mean()),mean_active=float((z>1e-4).float().sum(1).mean()),dead=int((z.max(0).values<1e-4).sum()),recovery=float(matched.mean()),intervention_alignment=alignment)
          records.append(rec);traces.append(dict(seed=seed,penalty=penalty,history=history))
          if seed==0 and penalty==.05:best=dict(cos=cos.tolist(),true_feature=d[0].tolist(),decoder_intervention=intervention.tolist(),matched_units=cc.tolist())
    return dict(records=records,traces=traces,example=best,seconds=time.perf_counter()-start)

## Experiment and checks

Assertions check correctness before the comparison. The sparse-autoencoder experiment uses held-out activations, and DPO evaluates fresh preference labels on known contexts. Training settings are deliberately small.

In [ ]:
result = sae_experiment()
print('Elapsed seconds:', result['seconds'])
print(result.get('errors', result['records'][0]))

## Measured figures

Each point comes from the experiment above. For attention on CPU the memory panel is labelled analytical score storage, because CPU allocator peaks are not measured. GPU results use allocator statistics.

In [ ]:
def sae_plots(result):
    fig,axes=plt.subplots(1,3,figsize=(13,4))
    for penalty in [0.,.01,.05,.15]:
      rows=[r for r in result['records'] if r['penalty']==penalty]
      axes[0].scatter([r['mean_active'] for r in rows],[r['mse'] for r in rows],label=str(penalty))
      axes[1].scatter([penalty]*len(rows),[r['recovery'] for r in rows])
    axes[0].set(xlabel='Mean active units',ylabel='Held-out reconstruction MSE');axes[0].legend(title='Penalty')
    axes[1].set(xlabel='Sparsity penalty',ylabel='Matched mean absolute cosine',ylim=(0,1))
    im=axes[2].imshow(np.abs(result['example']['cos']),aspect='auto',vmin=0,vmax=1,cmap='viridis');axes[2].set(xlabel='Learned decoder column',ylabel='True latent feature',title='Seed 0, penalty 0.05');fig.colorbar(im,ax=axes[2],shrink=.7)
    fig.tight_layout();return fig

sae_plots(result)
plt.show()

## Decoder intervention

Subtracting one activation times its decoder column changes the reconstruction along that column exactly. Comparing the column with a planted feature checks its geometric alignment; this is not a downstream model behaviour intervention.

In [ ]:
def sae_intervention(result):
    fig,ax=plt.subplots(figsize=(7,4));x=np.arange(8);a=np.array(result['example']['true_feature']);b=np.array(result['example']['decoder_intervention']);b=b/np.linalg.norm(b);b=b*np.sign(a@b)
    ax.plot(x,a,'o-',label='True feature direction');ax.plot(x,b,'s--',label='Matched decoder direction (sign-aligned)');ax.set(xlabel='Activation coordinate',ylabel='Direction component');ax.legend();fig.tight_layout();return fig
sae_intervention(result)
plt.show()

## Exercises

1. Change feature frequency from 0.08 to 0.3 and measure recovery.
2. Compare width 12, 24 and 48 with a fixed training budget.
3. Remove the unit-norm projection and inspect decoder norms.
4. Test interventions in a downstream function before claiming semantic control.

## Reference and scope

Primary reference: https://transformer-circuits.pub/2023/monosemantic-features/index.html

This is a controlled teaching experiment. The attention implementation uses ordinary PyTorch operations; the autoencoder uses planted synthetic features; DPO uses categorical responses rather than a language model.